# 07 — Inference & Demo

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

The last mile: a saved artefact scoring new rows, and the interactive demo.

**What makes this production-shaped rather than a notebook trick:** a
`ModelBundle` carries the model, its preprocessor *and* its feature list
together, and `predict()` re-validates that contract on every call. The classic
production failure is not a bad model — it is a good model scored on a frame
whose column 7 changed meaning six months after training.

> Prerequisites: notebooks `00`–`06` (notebook 03 saves the bundles).

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/ePGD - MLDS IIT Bombay/C5 ML In Finanace/Data"
    )

In [ ]:
import numpy as np
import pandas as pd

from novafin.config import load_config
from novafin.data import load_dataset, make_feature_frame
from novafin.features import build_features
from novafin.predict import (
    available_bundles, load_bundle, predict,
    score_credit_applications, score_customers, score_transactions,
)
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme

cfg = load_config()
setup_logging("INFO")
seed_everything(cfg.reproducibility.seed)
apply_theme()
pd.set_option("display.width", 200)

bundles = available_bundles(cfg)
print(f"saved bundles ({len(bundles)}):")
for path in bundles:
    print("   ", path.name)
if not bundles:
    print("   none - run notebooks/03_baseline_models.ipynb first")

## 2 · The feature contract

The bundle's sidecar `.json` records everything needed to trust the artefact:
the feature list, the cross-validated metrics, the config fingerprint and the
SHA-256 of every input file. It is readable without unpickling anything.

In [ ]:
if bundles:
    bundle = load_bundle(bundles[0], cfg)
    print(f"bundle          : {bundles[0].name}")
    print(f"dataset         : {bundle.dataset_key}")
    print(f"target          : {bundle.target_name}")
    print(f"features        : {len(bundle.feature_names)}")
    print(f"fingerprint     : {bundle.config_fingerprint}")
    print(f"trained (UTC)   : {bundle.created_utc}")
    print(f"model class     : {type(bundle.model).__name__}")
    print("\ncross-validated metrics recorded in the bundle:")
    for name, value in list(bundle.metrics.items())[:8]:
        print(f"   {name:<20} {value:.5f}")
    print("\ninput provenance:")
    for name, digest in list(bundle.data_hashes.items())[:4]:
        print(f"   {name:<28} {digest[:16]}")

In [ ]:
# The guard that earns the bundle its keep: a frame missing a training
# feature is REJECTED, not scored.
if bundles:
    try:
        bundle.validate_frame(["Credit_Score", "Debt_to_Income"])
    except ValueError as exc:
        print("REJECTED as expected:")
        print("   ", str(exc)[:160])

    shuffled = list(reversed(bundle.feature_names))
    restored = bundle.validate_frame(shuffled)
    print("\nColumn ORDER is restored automatically:")
    print("   passed in:", shuffled[:3], "...")
    print("   scored as:", restored[:3], "...")

## 3 · Credit — score applications to a decision

Not a probability: a **decision**. The rule uses PD, the ECL-to-exposure ratio
and collateral coverage together, because PD alone is not a lending decision.

In [ ]:
credit_bundle_path = cfg.paths.artifacts / "loans_baseline.pkl"
if credit_bundle_path.exists():
    credit_bundle = load_bundle(credit_bundle_path, cfg)
    loans_raw = load_dataset("loans", cfg=cfg).frame
    built = build_features("loans", loans_raw, cfg)
    X_loans, _ = make_feature_frame(built.frame, cfg.dataset("loans"))

    applications = X_loans.head(200).copy()
    applications["Loan_Amount"] = built.frame["Loan_Amount"].head(200).to_numpy()
    applications["Collateral_Value"] = built.frame["Collateral_Value"].head(200).to_numpy()
    applications["Customer_ID"] = built.frame["Customer_ID"].head(200).to_numpy() \
        if "Customer_ID" in built.frame.columns else range(200)

    scored = score_credit_applications(applications, credit_bundle, cfg=cfg)
    display(scored.head(10).round(4))

    print("\nDecision mix:")
    print(scored["decision"].value_counts().to_string())
    print("\nBy risk band:")
    display(scored.groupby("risk_band", observed=True).agg(
        n=("pd", "size"), mean_pd=("pd", "mean"),
        total_exposure=("exposure", "sum"), total_ecl=("ecl", "sum")).round(4))
else:
    print("No credit bundle found - run notebook 03 first.")

## 4 · Fraud — score at the cost-optimal threshold

The threshold comes from the cost curve, not from 0.5. At a 2.28% base rate the
cost-minimising operating point is nowhere near the default.

In [ ]:
fraud_bundle_path = cfg.paths.artifacts / "transactions_baseline.pkl"
if fraud_bundle_path.exists():
    fraud_bundle = load_bundle(fraud_bundle_path, cfg)
    txn_raw = load_dataset("transactions", cfg=cfg).frame
    txn_built = build_features("transactions", txn_raw, cfg)
    X_txn, _ = make_feature_frame(txn_built.frame, cfg.dataset("transactions"))
    X_txn = X_txn.drop(columns=[c for c in X_txn.columns
                                if pd.api.types.is_datetime64_any_dtype(X_txn[c])],
                       errors="ignore")

    # Use the threshold measured in notebook 06, if it was persisted.
    threshold = None
    holdout_path = cfg.paths.tables / "06_fraud_holdout.csv"
    if (cfg.paths.tables / "03_baseline_summary.csv").exists():
        threshold = fraud_bundle.metadata.get("optimal_threshold")

    scored_txn = score_transactions(X_txn.head(500), fraud_bundle, cfg=cfg,
                                    threshold=threshold or 0.15)
    display(scored_txn.head(10).round(4))
    print("\nPriority mix:")
    print(scored_txn["priority"].value_counts().to_string())
    print(f"\nflagged {scored_txn['investigate'].sum()} of {len(scored_txn)} "
          f"at threshold {scored_txn['threshold'].iloc[0]:.3f}")
else:
    print("No fraud bundle found - run notebook 03 first.")

## 5 · Customers — the value-based answer

`score_customers` takes an **optional** bundle. Finding **N-01** established
there is no learnable churn signal here, so with no bundle the ranking is
value-based (CLV × complaints × disengagement) — which is the defensible answer
to *"which 1,000 customers do we contact?"*

In [ ]:
customers_raw = load_dataset("customers", cfg=cfg).frame
prioritised = score_customers(customers_raw, bundle=None, cfg=cfg)

display(prioritised.head(10).round(3))
print("\nbasis:", prioritised["basis"].iloc[0])
print("\nSegment mix:")
print(prioritised["segment"].value_counts().to_string())

contacted = prioritised[prioritised["contact"]]
print(f"\nTop {len(contacted):,} customers to contact:")
print(f"   mean CLV            : {contacted['clv'].mean():,.0f} "
      f"(book average {prioritised['clv'].mean():,.0f})")
print(f"   share of book value : "
      f"{contacted['clv'].sum() / prioritised['clv'].sum():.1%} "
      f"from {len(contacted) / len(prioritised):.0%} of customers")

## 6 · The Gradio demo

Four tabs: credit, fraud, the ₹1,000 crore allocation (re-optimised live), and
model governance. `share=True` produces a public URL with **no account and no
API key**, which is why Gradio was chosen over Streamlit.

The governance tab is deliberate: a decision engine that hides its caveats is
the wrong engine.

In [ ]:
# Launch it. In Colab this prints a public share link.
#
#   !python app.py
#
# Or inline:
from app import build_interface

demo = build_interface()
demo.launch(share=True, show_error=True)

---

## Phase 8 summary

| Deliverable | Status |
|---|---|
| `ModelBundle` round-trip with contract validation | ✅ |
| Credit scoring → Approve / Review / Reject | ✅ |
| Fraud scoring at the cost-optimal threshold | ✅ |
| Customer prioritisation (value-based, per N-01) | ✅ |
| Gradio demo, four tabs, no keys required | ✅ |

**The project is now complete end to end:** raw CSV → validated load →
causal features → cross-validated model → tuned → fine-tuned → explained →
priced in rupees → a Board recommendation → an interactive demo.